In [3]:
import pandas as pd

drivers = pd.read_csv(r"D:\Codenera\DA Project\archive (1)\drivers.csv")

drivers.head()

,driver_id,name,vehicle_id,rating,total_rides,available
0,1,Virginia Johnson,284,3.44,30,True
1,2,Daniel Maddox,18,2.59,61,False
2,3,Patricia Salazar,298,4.93,75,True
3,4,Cameron Hinton DDS,49,2.59,12,True
4,5,Jennifer Ayala,138,4.69,71,True


In [4]:
drivers.shape

(300, 6)

In [5]:
drivers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   driver_id    300 non-null    int64  
 1   name         300 non-null    object 
 2   vehicle_id   300 non-null    int64  
 3   rating       300 non-null    float64
 4   total_rides  300 non-null    int64  
 5   available    300 non-null    bool   
dtypes: bool(1), float64(1), int64(3), object(1)
memory usage: 12.1+ KB


In [6]:
drivers.columns

Index(['driver_id', 'name', 'vehicle_id', 'rating', 'total_rides',
       'available'],
      dtype='object')

In [7]:
drivers.duplicated().sum()

np.int64(0)

In [8]:
drivers['driver_id'].duplicated().sum()

np.int64(0)

In [9]:
drivers['rating'].describe()

count    300.000000
mean       2.975867
std        1.189947
min        1.010000
25%        1.860000
50%        2.955000
75%        4.032500
max        5.000000
Name: rating, dtype: float64

In [10]:
(drivers['total_rides'] < 0).sum()

np.int64(0)

In [11]:
drivers['available'].value_counts()

available
False    155
True     145
Name: count, dtype: int64

In [12]:
#rides

In [13]:
rides = pd.read_csv(r"D:\Codenera\DA Project\archive (1)\rides.csv")

In [14]:
rides.head()

,ride_id,user_id,start_location,end_location,ride_start_time,ride_end_time,distance_km,fare_amount,driver_id
0,1,4067,Larryborough,Norrisborough,2024-04-30 10:31:56,2024-04-30 11:05:56,2.03,3.33,145
1,2,9160,Williamston,Robertburgh,2024-09-11 12:07:26,2024-09-11 13:34:26,13.66,30.03,244
2,3,1990,Howeshire,Christopherland,2024-04-30 10:21:25,2024-04-30 10:40:25,5.50,15.67,234
3,4,1268,North Annton,East Heather,2024-04-12 04:06:58,2024-04-12 05:07:58,31.74,53.90,145
4,5,7296,South Joseph,Lake Samuelberg,2024-04-26 16:45:45,2024-04-26 18:31:45,13.54,38.40,39


In [15]:
rides.shape

(50000, 9)

In [16]:
# Count actual rides for each driver
actual_rides = rides.groupby('driver_id')['ride_id'].count().reset_index()

# Rename the column
actual_rides.rename(columns={'ride_id': 'actual_rides'}, inplace=True)

# Compare both datasets
comparison = drivers[['driver_id', 'total_rides']].merge(
    actual_rides,
    on='driver_id',
    how='left'
)

# Replace missing ride counts with 0
comparison['actual_rides'] = comparison['actual_rides'].fillna(0)

# Check the differences
comparison['difference'] = (
    comparison['total_rides'] - comparison['actual_rides']
)

comparison.head(10)

,driver_id,total_rides,actual_rides,difference
0,1,30,179,-149
1,2,61,177,-116
2,3,75,182,-107
3,4,12,164,-152
4,5,71,167,-96
5,6,164,156,8
6,7,23,178,-155
7,8,127,158,-31
8,9,117,141,-24
9,10,85,172,-87


In [17]:
(comparison['difference'] != 0).sum()

np.int64(298)

In [18]:
print("Recorded rides greater than actual:",
      (comparison['difference'] > 0).sum())

print("Recorded rides less than actual:",
      (comparison['difference'] < 0).sum())

print("Matching ride counts:",
      (comparison['difference'] == 0).sum())

Recorded rides greater than actual: 45
Recorded rides less than actual: 253
Matching ride counts: 2


In [19]:
# Check whether ride records contain unknown driver IDs

invalid_drivers = rides[
    ~rides['driver_id'].isin(drivers['driver_id'])
]

print("Invalid driver IDs:", invalid_drivers['driver_id'].nunique())

print("Ride records with invalid driver IDs:", len(invalid_drivers))

Invalid driver IDs: 0
Ride records with invalid driver IDs: 0


In [20]:
rides.isnull().sum()

ride_id            0
user_id            0
start_location     0
end_location       0
ride_start_time    0
ride_end_time      0
distance_km        0
fare_amount        0
driver_id          0
dtype: int64

In [21]:
rides.isnull().sum().sum()

np.int64(0)

In [22]:
rides.duplicated().sum()

np.int64(0)

In [23]:
rides['ride_id'].duplicated().sum()

np.int64(0)

In [24]:
# Check zero or negative distances
(rides['distance_km'] <= 0).sum()

np.int64(0)

In [25]:
# Check zero or negative fares
(rides['fare_amount'] <= 0).sum()

np.int64(0)

In [26]:
# Check basic statistical values
rides[['distance_km', 'fare_amount']].describe()

,distance_km,fare_amount
count,50000.000000,50000.000000
mean,25.535241,57.431624
std,14.156627,34.260297
min,1.000000,1.570000
25%,13.220000,28.670000
50%,25.540000,55.290000
75%,37.890000,82.010000
max,50.000000,149.130000


In [27]:
print(rides[['ride_start_time', 'ride_end_time']].dtypes)

ride_start_time    object
ride_end_time      object
dtype: object


In [28]:
start = pd.to_datetime(rides['ride_start_time'], errors='coerce')
end = pd.to_datetime(rides['ride_end_time'], errors='coerce')

print("Invalid start dates:", start.isna().sum())
print("Invalid end dates:", end.isna().sum())

Invalid start dates: 0
Invalid end dates: 0


In [29]:
(end < start).sum()

np.int64(0)

In [30]:
duration = end - start

print("Zero-duration rides:", (duration == pd.Timedelta(0)).sum())

print("Negative-duration rides:", (duration < pd.Timedelta(0)).sum())

Zero-duration rides: 0
Negative-duration rides: 0


In [31]:
users = pd.read_csv(r"D:\Codenera\DA Project\archive (1)\users.csv")

ratings = pd.read_csv(r"D:\Codenera\DA Project\archive (1)\ratings.csv")

vehicles = pd.read_csv(r"D:\Codenera\DA Project\archive (1)\vehicles.csv")

In [32]:
# 1. BASIC DATA QUALITY CHECKS

datasets = {
    'drivers': drivers,
    'rides': rides,
    'users': users,
    'ratings': ratings,
    'vehicles': vehicles
}

for name, df in datasets.items():
    print("\n==========", name.upper(), "==========")
    print("Shape:", df.shape)
    print("Missing values:", df.isnull().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())


# 2. PRIMARY KEY CHECKS

primary_keys = {
    'drivers': ('drivers', 'driver_id'),
    'rides': ('rides', 'ride_id'),
    'users': ('users', 'user_id'),
    'ratings': ('ratings', 'rating_id'),
    'vehicles': ('vehicles', 'vehicle_id')
}

for name, (table, column) in primary_keys.items():
    print(name, "duplicate IDs:",
          datasets[table][column].duplicated().sum())


# 3. FOREIGN KEY CHECKS

print("\n========== FOREIGN KEY CHECKS ==========")

print("Invalid ride driver IDs:",
      (~rides['driver_id'].isin(drivers['driver_id'])).sum())

print("Invalid ride user IDs:",
      (~rides['user_id'].isin(users['user_id'])).sum())

print("Invalid driver vehicle IDs:",
      (~drivers['vehicle_id'].isin(vehicles['vehicle_id'])).sum())

print("Invalid rating ride IDs:",
      (~ratings['ride_id'].isin(rides['ride_id'])).sum())

print("Invalid rating user IDs:",
      (~ratings['user_id'].isin(users['user_id'])).sum())


# 4. NUMERIC VALIDATION

print("\n========== NUMERIC CHECKS ==========")

print("Invalid driver ratings:",
      ((drivers['rating'] < 1) | (drivers['rating'] > 5)).sum())

print("Negative driver rides:",
      (drivers['total_rides'] < 0).sum())

print("Invalid ride distances:",
      (rides['distance_km'] <= 0).sum())

print("Invalid ride fares:",
      (rides['fare_amount'] <= 0).sum())

print("Invalid customer ratings:",
      ((ratings['rating_value'] < 1) |
       (ratings['rating_value'] > 5)).sum())

print("Invalid vehicle capacity:",
      (vehicles['capacity'] <= 0).sum())


# 5. DATE VALIDATION

print("\n========== DATE CHECKS ==========")

start = pd.to_datetime(rides['ride_start_time'], errors='coerce')
end = pd.to_datetime(rides['ride_end_time'], errors='coerce')

print("Invalid ride start dates:", start.isna().sum())
print("Invalid ride end dates:", end.isna().sum())
print("End before start:", (end < start).sum())

registration = pd.to_datetime(
    users['registration_date'], errors='coerce'
)

print("Invalid registration dates:", registration.isna().sum())


# 6. RATINGS CONSISTENCY CHECK

print("\n========== RATINGS CONSISTENCY ==========")

rating_check = ratings.merge(
    rides[['ride_id', 'user_id', 'ride_start_time']],
    on='ride_id',
    suffixes=('_rating', '_ride')
)

print("Rating user mismatches:",
      (rating_check['user_id_rating'] !=
       rating_check['user_id_ride']).sum())

rating_dates = pd.to_datetime(
    rating_check['rating_date'], errors='coerce'
)

ride_dates = pd.to_datetime(
    rating_check['ride_start_time'], errors='coerce'
)

print("Invalid rating dates:", rating_dates.isna().sum())

print("Ratings before ride:",
      (rating_dates < ride_dates).sum())


========== DRIVERS ==========
Shape: (300, 6)
Missing values: 0
Duplicate rows: 0

========== RIDES ==========
Shape: (50000, 9)
Missing values: 0
Duplicate rows: 0

========== USERS ==========
Shape: (10000, 8)
Missing values: 0
Duplicate rows: 0

========== RATINGS ==========
Shape: (50000, 6)
Missing values: 25103
Duplicate rows: 0

========== VEHICLES ==========
Shape: (300, 5)
Missing values: 0
Duplicate rows: 0
drivers duplicate IDs: 0
rides duplicate IDs: 0
users duplicate IDs: 0
ratings duplicate IDs: 0
vehicles duplicate IDs: 0

========== FOREIGN KEY CHECKS ==========
Invalid ride driver IDs: 0
Invalid ride user IDs: 0
Invalid driver vehicle IDs: 0
Invalid rating ride IDs: 0
Invalid rating user IDs: 0

========== NUMERIC CHECKS ==========
Invalid driver ratings: 0
Negative driver rides: 0
Invalid ride distances: 0
Invalid ride fares: 0
Invalid customer ratings: 0
Invalid vehicle capacity: 0

========== DATE CHECKS ==========
Invalid ride start dates: 0
Invalid ride end dates

In [33]:
ratings.isnull().sum()

rating_id           0
ride_id             0
user_id             0
rating_value        0
comments        25103
rating_date         0
dtype: int64

In [34]:
ratings_clean = ratings.copy()

ratings_clean['comments'] = ratings_clean['comments'].fillna('No comment')

In [35]:
ratings_clean.isnull().sum()

rating_id       0
ride_id         0
user_id         0
rating_value    0
comments        0
rating_date     0
dtype: int64

In [36]:
ratings_clean.shape

(50000, 6)

In [37]:
# Create cleaned copies of all datasets

drivers_clean = drivers.copy()
rides_clean = rides.copy()
users_clean = users.copy()
vehicles_clean = vehicles.copy()

# ratings_clean already exists

In [38]:
# Convert ride dates
rides_clean['ride_start_time'] = pd.to_datetime(
    rides_clean['ride_start_time']
)

rides_clean['ride_end_time'] = pd.to_datetime(
    rides_clean['ride_end_time']
)

# Convert user registration date
users_clean['registration_date'] = pd.to_datetime(
    users_clean['registration_date']
)

# Convert rating date
ratings_clean['rating_date'] = pd.to_datetime(
    ratings_clean['rating_date']
)

In [39]:
import os

os.makedirs("cleaned_data", exist_ok=True)

drivers_clean.to_csv("cleaned_data/drivers_clean.csv", index=False)
rides_clean.to_csv("cleaned_data/rides_clean.csv", index=False)
users_clean.to_csv("cleaned_data/users_clean.csv", index=False)
ratings_clean.to_csv("cleaned_data/ratings_clean.csv", index=False)
vehicles_clean.to_csv("cleaned_data/vehicles_clean.csv", index=False)

In [1]:
import os
print(os.getcwd())

C:\Users\adity\Theta A\DA Project


In [1]:
print(drivers_clean.shape)

print(drivers_clean.head())

print(drivers_clean.dtypes)

NameError: name 'drivers_clean' is not defined

In [2]:
import pandas as pd

drivers_clean = pd.read_csv(
    r"C:\Users\adity\Theta A\DA Project\cleaned_data\drivers_clean.csv"
)

vehicles_clean = pd.read_csv(
    r"C:\Users\adity\Theta A\DA Project\cleaned_data\vehicles_clean.csv"
)

print("Drivers:", drivers_clean.shape)
print("Vehicles:", vehicles_clean.shape)

print(drivers_clean.head())
print(drivers_clean.dtypes)

Drivers: (300, 6)
Vehicles: (300, 5)
   driver_id                name  vehicle_id  rating  total_rides  available
0          1    Virginia Johnson         284    3.44           30       True
1          2       Daniel Maddox          18    2.59           61      False
2          3    Patricia Salazar         298    4.93           75       True
3          4  Cameron Hinton DDS          49    2.59           12       True
4          5      Jennifer Ayala         138    4.69           71       True
driver_id        int64
name            object
vehicle_id       int64
rating         float64
total_rides      int64
available         bool
dtype: object


In [3]:
print(drivers_clean['vehicle_id'].isin(vehicles_clean['vehicle_id']).all())

True


In [4]:
drivers_mysql = drivers_clean.copy()

drivers_mysql['available'] = drivers_mysql['available'].astype(int)

drivers_mysql.to_csv(
    r"C:\Users\adity\Theta A\DA Project\cleaned_data\drivers_mysql.csv",
    index=False
)

print(drivers_mysql.head())
print(drivers_mysql.shape)

   driver_id                name  vehicle_id  rating  total_rides  available
0          1    Virginia Johnson         284    3.44           30          1
1          2       Daniel Maddox          18    2.59           61          0
2          3    Patricia Salazar         298    4.93           75          1
3          4  Cameron Hinton DDS          49    2.59           12          1
4          5      Jennifer Ayala         138    4.69           71          1
(300, 6)


In [5]:
users_clean.shape

NameError: name 'users_clean' is not defined

In [6]:
import pandas as pd

users_clean = pd.read_csv(
    r"C:\Users\adity\Theta A\DA Project\cleaned_data\users_clean.csv"
)

print(users_clean.shape)
print(users_clean.head())
print(users_clean.dtypes)

(10000, 8)
   user_id               name                          email  \
0        1        Tammy Jones  trevinojacqueline@example.net   
1        2      Tyler Mathews              tim47@example.com   
2        3  Melissa Robertson            scott64@example.org   
3        4       Heather Wolf         rushandrew@example.net   
4        5      Daniel Daniel           robert13@example.org   

        phone_number    registration_date  age  gender           location  
0   530-318-6905x290  2024-07-29 12:00:05   21   Other       New Alanside  
1  (222)541-3246x834  2021-10-04 14:21:09   53    Male   New Virginiabury  
2   309-644-3534x739  2022-10-07 04:46:44   57    Male         Ethanmouth  
3  998-772-1220x9334  2022-07-10 14:01:22   50    Male  South Theresaland  
4    +1-992-853-1435  2020-05-21 04:37:46   32  Female          New James  
user_id               int64
name                 object
email                object
phone_number         object
registration_date    object
age     

In [7]:
users_mysql = users_clean.copy()

users_mysql['registration_date'] = pd.to_datetime(
    users_mysql['registration_date']
)

users_mysql.to_csv(
    r"C:\Users\adity\Theta A\DA Project\cleaned_data\users_mysql.csv",
    index=False
)

print(users_mysql.shape)

(10000, 8)
